In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle

from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    feature_cols = ["symbol_id", "time_id"] \
        + [f"feature_{idx:02d}" for idx in range(79)] \
        + [f"responder_{idx}_lag_1" for idx in range(9)]
    categorical_cols = []

In [10]:
training_data = pl.scan_parquet("train.parquet").collect()
valid = pl.scan_parquet("valid.parquet").collect()
cols = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)] \
+ [f'symbol_id_{ind}' for ind in range(39)]
# one-hot encoding for symbol_id
training = training_data.with_columns(
    [
        (pl.col("symbol_id") == idx).cast(pl.Int8).alias(f"symbol_id_{idx}")
        for idx in range(39)
    ]
).drop('symbol_id')

# forward fill na
training = training.with_columns(
    [
        pl.col(column).fill_null(strategy='forward').alias(column)
        for column in cols
    ]
)


condition = pl.lit(True)
for column in cols:
    condition &= pl.col(column).is_not_null()

training = training.filter(condition)
type(training)

polars.dataframe.frame.DataFrame

In [11]:
validation = valid.with_columns(
    [
        (pl.col("symbol_id") == idx).cast(pl.Int8).alias(f"symbol_id_{idx}")
        for idx in range(39)
    ]
).drop('symbol_id')

# forward fill na
validation = validation.with_columns(
    [
        pl.col(column).fill_null(strategy='forward').alias(column)
        for column in cols
    ]
)

validation = validation.filter(condition)
validation

id,date_id,time_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,…,symbol_id_2,symbol_id_3,symbol_id_4,symbol_id_5,symbol_id_6,symbol_id_7,symbol_id_8,symbol_id_9,symbol_id_10,symbol_id_11,symbol_id_12,symbol_id_13,symbol_id_14,symbol_id_15,symbol_id_16,symbol_id_17,symbol_id_18,symbol_id_19,symbol_id_20,symbol_id_21,symbol_id_22,symbol_id_23,symbol_id_24,symbol_id_25,symbol_id_26,symbol_id_27,symbol_id_28,symbol_id_29,symbol_id_30,symbol_id_31,symbol_id_32,symbol_id_33,symbol_id_34,symbol_id_35,symbol_id_36,symbol_id_37,symbol_id_38
u32,i16,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
46047698,1670,68,3.405333,0.486807,1.307911,0.257714,0.192415,1.232755,-0.409265,-0.792289,-0.952887,-0.038853,11,7,76,0.320592,0.412181,0.962468,-0.907869,-0.689321,-0.285065,-0.396515,-0.721884,0.752113,-0.128356,1.485508,0.798556,0.939105,0.531187,1.102988,0.510059,0.409313,-0.711646,-0.902749,-0.162899,0.681765,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
46047699,1670,68,2.619899,0.056043,0.730986,0.285026,0.378605,1.459118,-0.727881,-1.004993,-0.79925,-0.036213,11,7,76,0.32165,0.130736,0.424283,-0.682356,-0.43819,-0.692322,-0.125519,-0.167758,1.357302,-0.007474,1.107871,0.374563,1.059325,1.151365,-1.985968,-0.465673,0.825128,-0.756643,-0.735251,-0.006619,0.041776,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
46047700,1670,68,2.213198,0.454587,1.559841,0.917671,0.235877,1.580666,-0.747848,-0.704533,-1.088991,-0.04975,81,2,59,-0.853819,-0.437449,-0.619555,-0.703064,-0.27165,-0.339336,-0.736839,-0.115122,0.740504,-0.174841,0.804582,1.037138,0.885568,-0.109851,0.672542,0.068193,-0.324793,-0.46759,-0.299015,-0.20314,-0.179034,…,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
46047701,1670,68,1.603719,0.055734,1.362784,0.061581,0.327499,1.399431,-0.669743,-1.246755,-0.890678,-0.050033,4,3,11,-0.298021,-0.119403,0.030537,-0.573521,-0.384419,-0.707569,-0.059272,-1.846481,-0.116152,-0.155462,0.158984,0.268396,0.917793,0.372768,-0.010643,-0.451999,-0.813563,-0.64412,-0.543663,-0.165089,0.241096,…,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
46047702,1670,68,2.820571,-0.114807,0.926922,0.198282,0.140551,1.420192,-0.661859,-0.710849,-0.976321,-0.057802,2,10,171,0.046459,-0.357021,0.320223,-0.711042,-0.699025,-0.889487,-0.655191,-0.344973,-0.013563,-0.147847,0.963908,0.065798,0.948555,0.592319,-1.126012,-0.218446,0.668718,-0.822269,-0.703427,-0.116816,1.847894,…,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
47127333,1698,967,3.242493,2.52516,-0.721981,2.544025,2.477615,0.417557,0.785812,1.117796,2.199436,0.415427,42,5,150,0.804403,1.157257,1.031543,-0.671189,-0.3286,-0.486132,1.730176,-0.006173,-0.001144,-0.213062,0.932618,1.367338,-0.238197,-0.692615,-0.121163,1.090798,1.444294,-0.675626,-1.013264,-0.242888,3.427639,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
47127334,1698,967,1.079139,1.857906,-0.790646,2.745439,2.339877,0.845065,0.65137,1.180301,1.966379,0.321543,25,7,195,-0.075294,-0.152726,-0.20417,-0.421137,0.21708,-0.258775,1.874978,0.19988,-0.199219,-0.125619,-1.004547,-0.051933,0.450905,0.009246,0.164127,-0.939974,-1.143421,-0.320071,-0.379835,-0.142429,3.862469,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

In [14]:
train = training.to_pandas()
valid = validation.to_pandas()
train.shape, valid.shape

((20984304, 142), (1079640, 142))

In [ ]:
cols = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)] \
+ [f'symbol_id_{ind}' for ind in range(39)]

Index(['id', 'date_id', 'time_id', 'symbol_id', 'weight', 'feature_00',
       'feature_01', 'feature_02', 'feature_03', 'feature_04',
       ...
       'label', 'responder_0_lag_1', 'responder_1_lag_1', 'responder_2_lag_1',
       'responder_3_lag_1', 'responder_4_lag_1', 'responder_5_lag_1',
       'responder_6_lag_1', 'responder_7_lag_1', 'responder_8_lag_1'],
      dtype='object', length=104)

In [16]:
feature_names = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)] + [f'symbol_id_{ind}' for ind in range(39)]
train_X = []
for feature in feature_names:
    column = train[feature].to_numpy()  # Convert each column to NumPy array
    train_X.append(column)

train_X = np.column_stack(train_X)
import gc
gc.collect()

print(train_X)

[[-0.9241074  -0.00565404 -0.3515969  ...  0.          0.
   0.        ]
 [-0.15975541  0.14542861 -0.547485   ...  0.          0.
   0.        ]
 [ 0.11812775  0.33371982 -0.24753632 ...  0.          0.
   0.        ]
 ...
 [ 0.04255923 -1.7481911   0.2115967  ...  1.          0.
   0.        ]
 [ 0.01478909 -1.9616086   0.35203713 ...  0.          1.
   0.        ]
 [ 0.17674166 -1.4678111   0.8221977  ...  0.          0.
   1.        ]]


In [17]:
valid_X = []
for feature in feature_names:
    column = valid[feature].to_numpy()  # Convert each column to NumPy array
    valid_X.append(column)

# Stack the arrays to form the final 2D array
valid_X = np.column_stack(valid_X)

print(valid_X)

[[ 0.48680663  1.3079113   0.25771397 ...  0.          0.
   0.        ]
 [ 0.05604257  0.73098624  0.28502578 ...  0.          0.
   0.        ]
 [ 0.45458692  1.5598414   0.91767097 ...  0.          0.
   0.        ]
 ...
 [ 2.515527   -0.6722982   2.28925    ...  1.          0.
   0.        ]
 [ 2.6632981  -0.88911206  2.3131552  ...  0.          1.
   0.        ]
 [ 2.7285063  -0.7452382   2.7887886  ...  0.          0.
   1.        ]]


In [19]:
train_weight = train['weight'].to_numpy()
valid_weight = valid['weight'].to_numpy()
train_y = train['responder_6'].to_numpy()
valid_y = valid['responder_6'].to_numpy()

train_X.shape, valid_X.shape, train_weight.shape, valid_weight.shape, train_y.shape, valid_y.shape

((20984304, 127),
 (1079640, 127),
 (20984304,),
 (1079640,),
 (20984304,),
 (1079640,))

In [20]:
def get_model(seed):
    # XGBoost parameters
    XGB_Params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'n_estimators': 200,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 1,
        'reg_lambda': 5,
        'random_state': seed,
        'tree_method': 'gpu_hist',
        'device' : 'cuda',
        'n_gpus' : 2,
    }
    
    XGB_Model = XGBRegressor(**XGB_Params)
    return XGB_Model

In [21]:
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2

In [22]:
with open('ridge.pkl', 'rb') as f:
    model_ridge = pickle.load(f)

y_pred_ridge = model_ridge.predict(train_X)
# r2_val(y_train, y_pred_ridge, w_train), r2_score(y_train, y_pred_ridge, sample_weight=w_train)
r2_val(train_y, y_pred_ridge, train_weight), r2_score(train_y, y_pred_ridge, sample_weight=train_weight)


(0.006459113565989827, 0.0064522624015808105)

In [23]:
residuals = train_y - y_pred_ridge
# fit xgb on residuals
model_xgb = get_model(CONFIG.seed)
model_xgb.fit(train_X, residuals, sample_weight=train_weight)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_gpus=2, n_jobs=None,
             num_parallel_tree=None, ...)

In [24]:
y_pred_xgb = model_xgb.predict(train_X)
r2_val(train_y, y_pred_ridge + y_pred_xgb, train_weight), r2_score(train_y, y_pred_ridge + y_pred_xgb, sample_weight=train_weight)

(0.0344771193371759, 0.03447049856185913)

In [25]:
# fit both on valid
y_pred_ridge_valid = model_ridge.predict(valid_X)
y_pred_xgb_valid = model_xgb.predict(valid_X)
r2_val(valid_y, y_pred_ridge_valid + y_pred_xgb_valid, valid_weight), r2_score(valid_y, y_pred_ridge_valid + y_pred_xgb_valid, sample_weight=valid_weight)

(0.006576145162680236, 0.006554126739501953)